In [ ]:
from moviepy import AudioFileClip, VideoFileClip
from pydub import AudioSegment, effects
import pandas as pd
from yt_dlp import YoutubeDL
import csv
from natasha import (
    Segmenter,
    Doc
)
from faster_whisper import WhisperModel, BatchedInferencePipeline
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
batch_size = 32
compute_type = "float16"
model = WhisperModel("large-v3-turbo", device=device, compute_type=compute_type)
batched_model = BatchedInferencePipeline(model=model)

segmenter = Segmenter()

def split_by_sentences(text):
        doc = Doc(text)
        doc.segment(segmenter)

        return [x.text for x in doc.sents]


def words_to_sentences(words):
        sentences = []
        i = 0
        while i < len(words):
            stop = False
            j = i + 1
            while not stop and j <= len(words):
                clip = words[i:j]
                clip_text = ''.join([x['word'] for x in clip])
                clip_sentences = split_by_sentences(clip_text)
                if len(clip_sentences) > 1:
                    sentences.append({
                        "start": clip[0]['start'],
                        "end": clip[-1]['start'],
                        "text": clip_sentences[0]
                    })
                    i = j - 1
                    stop = True
                else:
                    j += 1

            if not stop and i < len(words):
                clip = words[i:]
                clip_text = ''.join([x['word'] for x in clip])
                sentences.append({
                    "start": clip[0]['start'],
                    "end": clip[-1]['end'],
                    "text": clip_text
                })
                break
        return sentences

def clip_filter(start, end, sentences):
    return list(filter(lambda x: start<=x['start'] and x['start'] <= end, sentences))  

In [ ]:
# !pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128
# !pip install faster-whisper
# !pip install moviepy
# !pip install natasha
# !pip install yt-dlp
# !pip install pydub
# !pip install pandas

In [ ]:
data = pd.read_csv("data_twitch.csv", delimiter=',')
data

In [ ]:
for video in data['video']:
    dataset = pd.DataFrame({'text': {}, 'label' : {}})
    print(video)
    #Скачиваем видео
    video_id = video.split("?v=")[1]
    ydl_opts = {
    'format': 'm4a/bestaudio/best',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'm4a', #mp3
    }],
    'outtmpl': f"audio_{video_id}"
    }
    YoutubeDL(ydl_opts).download(video)

    # segments, info = batched_model.transcribe(audio_file, batch_size=batch_size, vad_filter=True, vad_parameters=dict(min_silence_duration_ms=500))
    segments, _ = batched_model.transcribe(f'audio_{video_id}.m4a', batch_size=batch_size, word_timestamps=True)

    words = []
    for segment in segments:
        for word in segment.words:
            words.append({
                "start": word.start,
                "end": word.end,
                "word": word.word
            })
    
    sentences = words_to_sentences(words)

    tags = data[data['video'] == video]['tags'].values[0].replace('-', ' ').split(' ')
    sec_tags = []
    for tag in tags:
        min_sec = tag.split(':')
        if len(min_sec) == 3:
            sec_tags.append(int(min_sec[0])*60*60 + int(min_sec[1])*60 + int(min_sec[2]))
        else:
            sec_tags.append(int(min_sec[0])*60 + int(min_sec[1]))

    for i in range(len(sec_tags)-1):
        is_interesting = (i % 2 == 0)
        start, end = sec_tags[i], sec_tags[i+1]
        
        clip_sentences = clip_filter(start, end, sentences)

        rows_list = []
        for sentence in clip_sentences:
            rows_list.append({'text': sentence['text'], 'label': int(is_interesting)})

        dataset = pd.concat([dataset, pd.DataFrame(rows_list)]).reset_index(drop=True)
        dataset.to_csv(f'datasets_twitch/dataset_twitch_{video_id}.csv', index=False)
        